# Telugu Speech AI — Capstone Final Project
### Learn and Help | Python ML Course | Week 27 | Student: Ahala

**Project Goal:** Build a system that can:
1. Transcribe spoken Telugu using multiple models (India vs USA)
2. Compare model accuracy using WER and CER
3. Synthesize Telugu speech — even cloning a voice

Run each section in order. Make sure you're connected to a GPU runtime (Runtime > Change runtime type > T4 GPU).


## Setup — Install Libraries

In [ ]:
# Install all required libraries
!pip install openai-whisper --quiet
!pip install transformers datasets soundfile librosa jiwer --quiet
!pip install TTS --quiet          # Coqui TTS for voice cloning
!pip install gTTS --quiet         # Google TTS
!pip install huggingface_hub --quiet

print("All libraries installed!")


In [ ]:
import torch
import librosa
import soundfile as sf
import numpy as np
import os

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


---
## Task 1 — Data Collection and Setup

We'll use a few sample Telugu sentences. You can also record your own audio
or download from the OpenSLR Telugu dataset (https://openslr.org/66/).

**Sample Telugu sentences for testing:**
- naenu telugu neerchukunTunnanu  (I am learning Telugu)
- manchi roju (Good day)
- meeru ela unnaru (How are you)

If you want to record your own, use your phone or Audacity, then upload to Colab.


In [ ]:
# Download a sample Telugu audio file for testing
# We use a short clip from the AI4Bharat sample set

import urllib.request

os.makedirs("audio_samples", exist_ok=True)

# You can replace these URLs with your own uploaded files
# For demonstration, we create a synthetic test signal
from scipy.io.wavfile import write as wav_write

print("For this notebook, please upload your Telugu WAV files to the audio_samples/ folder.")
print("Then update the file list below with your actual filenames.")
print()
print("Expected format: 16kHz, mono, WAV")
print()

# Ground truth table — fill this in with YOUR audio files
audio_files = [
    {"file": "audio_samples/sentence1.wav", "ground_truth": ""},
    {"file": "audio_samples/sentence2.wav", "ground_truth": ""},
    {"file": "audio_samples/sentence3.wav", "ground_truth": ""},
]

print("Audio file table:")
for i, a in enumerate(audio_files, 1):
    print(f"  {i}. {a['file']} | Ground truth: {a['ground_truth'] or '(fill in)'}")


In [ ]:
# Helper: Preprocess audio to 16kHz mono WAV
def preprocess_audio(input_path, output_path=None):
    audio, sr = librosa.load(input_path, sr=16000, mono=True)
    if output_path is None:
        output_path = input_path.replace(".wav", "_16k.wav")
    sf.write(output_path, audio, 16000)
    print(f"Saved: {output_path} | Duration: {len(audio)/16000:.1f}s | SR: 16000 Hz")
    return output_path

# Visualize a waveform
import matplotlib.pyplot as plt

def plot_waveform(path, title="Audio Waveform"):
    audio, sr = librosa.load(path, sr=16000)
    plt.figure(figsize=(10, 2.5))
    plt.plot(np.linspace(0, len(audio)/sr, len(audio)), audio, color='steelblue', linewidth=0.5)
    plt.title(title, fontsize=12)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

print("Preprocessing helpers ready. Run preprocess_audio('your_file.wav') on each clip.")


---
## Task 2 — Speech-to-Text Model Comparison

We'll run 3 models on the same audio and compare their transcriptions and accuracy.

### Model 1: OpenAI Whisper (USA-origin)


In [ ]:
# Model 1: OpenAI Whisper
import whisper

print("Loading Whisper medium model...")
whisper_model = whisper.load_model("medium")

def run_whisper(audio_path):
    result = whisper_model.transcribe(audio_path, language="te")
    return result["text"]

# Test
test_file = audio_files[0]["file"]   # Change to your actual file
# transcription = run_whisper(test_file)
# print("Whisper output:", transcription)

print("Whisper model loaded. Call run_whisper('your_file.wav') to transcribe.")


### Model 2: Meta MMS (USA-origin)

In [ ]:
# Model 2: Meta MMS (Massively Multilingual Speech)
from transformers import pipeline, AutoProcessor, Wav2Vec2ForCTC
import torch

print("Loading Meta MMS model (supports 1,100+ languages)...")

mms_asr = pipeline(
    "automatic-speech-recognition",
    model="facebook/mms-300m",
    generate_kwargs={"language": "tel"},   # tel = Telugu language code
    device=0 if torch.cuda.is_available() else -1,
)

def run_mms(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)
    result = mms_asr({"array": audio, "sampling_rate": sr})
    return result["text"]

# test_output = run_mms(test_file)
# print("MMS output:", test_output)
print("Meta MMS loaded. Call run_mms('your_file.wav') to transcribe.")


### Model 3: AI4Bharat IndicWav2Vec (India-origin)

In [ ]:
# Model 3: AI4Bharat IndicWav2Vec (India-origin)
# This model was built specifically for Indian languages by IIT Madras + AI4Bharat

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

print("Loading AI4Bharat IndicWav2Vec for Telugu...")

indic_model_name = "Vakyansh/wav2vec2-telugu-tem-100"  # Vakyansh Telugu model
indic_processor  = Wav2Vec2Processor.from_pretrained(indic_model_name)
indic_model      = Wav2Vec2ForCTC.from_pretrained(indic_model_name).to(device)
indic_model.eval()

def run_indic(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = indic_processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = indic_model(inputs.input_values.to(device)).logits
    predicted_ids  = torch.argmax(logits, dim=-1)
    transcription  = indic_processor.batch_decode(predicted_ids)[0]
    return transcription

# test_output = run_indic(test_file)
# print("IndicWav2Vec output:", test_output)
print("AI4Bharat IndicWav2Vec loaded. Call run_indic('your_file.wav') to transcribe.")


### Compare All Models — WER and CER

In [ ]:
# Compute WER and CER for all models
from jiwer import wer, cer
import time

def evaluate_all_models(audio_path, ground_truth):
    results = {}

    print(f"Evaluating: {audio_path}")
    print(f"Ground truth: {ground_truth}")
    print()

    models = {
        "OpenAI Whisper (USA)": run_whisper,
        "Meta MMS (USA)":       run_mms,
        "AI4Bharat Vakyansh (India)": run_indic,
    }

    for name, fn in models.items():
        t_start = time.time()
        output = fn(audio_path)
        t_end  = time.time()
        duration, _ = librosa.load(audio_path, sr=16000)
        audio_len = len(duration) / 16000
        rtf = (t_end - t_start) / audio_len

        w = wer(ground_truth, output) * 100
        c = cer(ground_truth, output) * 100

        results[name] = {
            "output":   output,
            "WER (%)":  round(w, 1),
            "CER (%)":  round(c, 1),
            "RTF":      round(rtf, 2),
        }
        print(f"Model: {name}")
        print(f"  Output: {output}")
        print(f"  WER: {w:.1f}%  |  CER: {c:.1f}%  |  RTF: {rtf:.2f}")
        print()

    return results

# Run for all your test files:
# for item in audio_files:
#     res = evaluate_all_models(item["file"], item["ground_truth"])

print("Ready! Call evaluate_all_models(audio_path, ground_truth) for each clip.")


In [ ]:
# Build a summary comparison table
import pandas as pd

# After running evaluate_all_models on all clips, summarize here:

# Example structure — fill with your actual results:
summary_data = {
    "Model":     ["OpenAI Whisper (USA)", "Meta MMS (USA)", "AI4Bharat Vakyansh (India)"],
    "Origin":    ["USA",                  "USA",            "India"],
    "Avg WER %": [0.0,                    0.0,              0.0],   # fill in from your runs
    "Avg CER %": [0.0,                    0.0,              0.0],
    "Avg RTF":   [0.0,                    0.0,              0.0],
}

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

# Visualize
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df))
bars = ax.bar(df["Model"], df["Avg WER %"],
              color=["steelblue", "royalblue", "darkorange"],
              edgecolor="white", linewidth=1.5)
ax.set_ylabel("Average WER (%)")
ax.set_title("Model Comparison — Telugu ASR (Lower WER = Better)")
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()


---
## Task 3 — Text-to-Speech and Voice Cloning

### Option A: Voice Cloning with Coqui XTTS-v2

Record yourself speaking a Telugu sentence (10-30 seconds), upload as `my_voice.wav`.
Then XTTS-v2 will speak any new text in your voice!


In [ ]:
# Option A: Voice Cloning with Coqui XTTS-v2
from TTS.api import TTS

print("Loading XTTS-v2 voice cloning model...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

def clone_voice(text_telugu, speaker_wav, output_file="cloned_output.wav"):
    tts.tts_to_file(
        text=text_telugu,
        speaker_wav=speaker_wav,       # Your 10-30 sec recording
        language="te",                 # Telugu
        file_path=output_file,
    )
    print(f"Voice cloned audio saved to: {output_file}")
    return output_file

# Example usage (replace with your own voice recording):
# my_sample_text = "naenu telugu maatlaadutaanu"
# clone_voice(my_sample_text, "my_voice.wav", "cloned_output.wav")

print("XTTS-v2 loaded! Upload your voice recording as 'my_voice.wav' and call clone_voice().")


### Option B: Standard TTS Comparison (AI4Bharat IndicTTS vs Google TTS)

In [ ]:
# Option B: Compare AI4Bharat IndicTTS vs Google TTS
from gtts import gTTS
from IPython.display import Audio, display

def google_tts_telugu(text, output_file="gtts_output.wav"):
    tts = gTTS(text=text, lang='te')
    tts.save(output_file)
    print(f"Google TTS saved: {output_file}")
    return output_file

# Generate and play
test_text = "naenu telugu neerchukunTunnanu"
output = google_tts_telugu(test_text, "google_telugu.mp3")

# Play in Colab
from IPython.display import Audio
display(Audio("google_telugu.mp3"))


In [ ]:
# Rate the TTS outputs: MOS scoring
# Listen to each output above and rate it 1-5:
# 5 = sounds completely natural and human
# 4 = mostly natural, minor issues
# 3 = understandable but robotic
# 2 = difficult to understand
# 1 = very poor quality

mos_scores = {
    "Google TTS (gTTS)": None,        # Fill in: e.g., 3.5
    "XTTS-v2 Cloned":    None,        # Fill in after cloning
}

print("MOS Scores (fill in your ratings after listening):")
for model, score in mos_scores.items():
    display_score = str(score) if score else "(not rated yet)"
    print(f"  {model}: {display_score} / 5")

print()
print("After filling in scores, summarize your findings in the cell below.")


---
## Task 4 — Final Report

**Instructions:** Fill in each section below with your findings.

---

### 1. Introduction
*Why is Telugu ASR/TTS important? Who benefits from this technology?*

(Your answer here)

---

### 2. Methods
*Which models did you use and why?*

(Your answer here)

---

### 3. Results

Paste your comparison table (from Task 2) and MOS scores (from Task 3) here.

(Your results table here)

---

### 4. Key Finding
*Which model performed best on Telugu? Were India-origin models better? Why do you think that is?*

(Your answer here)

---

### 5. Connection to the Course
*Whisper and XTTS-v2 both use Transformers internally. How do Transformers help with audio?*
*Hint: Think about what "attention" means for audio sequences.*

(Your answer here)

---

### 6. What's Next?
*If you had more time, what would you try?*

(Your answer here)


---
## Bonus — Gradio Demo App

Build an interactive app where you can upload audio and see all 3 transcriptions!


In [ ]:
# BONUS: Gradio interface for live model comparison
!pip install gradio --quiet

import gradio as gr

def transcribe_all(audio_path):
    results = {}
    for name, fn in [
        ("Whisper (USA)",  run_whisper),
        ("Meta MMS (USA)", run_mms),
        ("Vakyansh (India)", run_indic),
    ]:
        try:
            results[name] = fn(audio_path)
        except Exception as e:
            results[name] = f"Error: {e}"
    output = ""
    for model, text in results.items():
        output += f"**{model}**\n{text}\n\n"
    return output

demo = gr.Interface(
    fn=transcribe_all,
    inputs=gr.Audio(type="filepath", label="Upload Telugu Audio"),
    outputs=gr.Markdown(label="Transcriptions from All 3 Models"),
    title="Telugu ASR — India vs USA Model Comparison",
    description="Upload a Telugu audio clip and see how different models transcribe it."
)

demo.launch(share=True)   # share=True gives a public URL


---
## Task 5 — Pricing Tier Reflection

Understanding *which* model to pick requires thinking beyond accuracy.
Cost, privacy, and ease of use all matter — especially in the real world.


### Step 1: Build Your Pricing Comparison Table

Fill in the `actual_wer` column from your Task 2 results.
The pricing data below is pre-filled from each provider's public pricing page (April 2025).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Fill in your actual WER results from Task 2:
your_wer_results = {
    "Whisper (OpenAI)":     None,   # e.g., 22.4
    "Meta MMS":             None,
    "AI4Bharat Vakyansh":   None,
    "Coqui XTTS-v2 (TTS)":  "N/A (TTS, not ASR)",
    "ElevenLabs":            "N/A (TTS, not ASR)",
    "Google Cloud STT":      None,  # only if you tested it
}

# Pricing data (public, as of 2025)
pricing_table = [
    {
        "Tool":         "Whisper (self-hosted)",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": your_wer_results.get("Whisper (OpenAI)", "?"),
    },
    {
        "Tool":         "Meta MMS",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": your_wer_results.get("Meta MMS", "?"),
    },
    {
        "Tool":         "AI4Bharat Vakyansh",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": your_wer_results.get("AI4Bharat Vakyansh", "?"),
    },
    {
        "Tool":         "Coqui XTTS-v2",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": "TTS only",
    },
    {
        "Tool":         "ElevenLabs",
        "Tier":         "Freemium",
        "Free Limit":   "10,000 chars/month",
        "Paid Cost":    "$22/mo (Starter)",
        "Privacy":      "Partial (cloud)",
        "Your WER (%)": "TTS only",
    },
    {
        "Tool":         "Google Cloud STT",
        "Tier":         "Paid",
        "Free Limit":   "60 min/month",
        "Paid Cost":    "$0.016/min",
        "Privacy":      "Low (Google servers)",
        "Your WER (%)": your_wer_results.get("Google Cloud STT", "not tested"),
    },
    {
        "Tool":         "AssemblyAI",
        "Tier":         "Freemium",
        "Free Limit":   "5 hrs audio free",
        "Paid Cost":    "$0.37/hr",
        "Privacy":      "Partial (cloud)",
        "Your WER (%)": "not tested",
    },
]

df = pd.DataFrame(pricing_table)
print(df.to_string(index=False))


### Step 2: Visualize Cost vs. Accuracy

In [ ]:
# Scatter plot: WER vs Cost (conceptual — use your own WER values)
# Only include ASR models with numeric WER values

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#F8F9FA')

# Replace these with your actual measured WER values
# x = approximate monthly cost for a hobbyist (~1 hr audio/month)
# y = your measured WER (%)
models_plot = [
    ("Whisper
(OpenAI)", 0,    22.4, '#16A34A', 'Free'),
    ("Meta MMS",          0,    31.8, '#16A34A', 'Free'),
    ("AI4Bharat
Vakyansh", 0,  14.2, '#16A34A', 'Free'),
    ("ElevenLabs
(ASR)", 0,    None, '#D97706', 'Freemium'),
    ("Google
Cloud STT", 0.96, 10.5, '#DC2626', 'Paid'),   # ~60 min * $0.016
]

tier_colors = {'Free': '#16A34A', 'Freemium': '#D97706', 'Paid': '#DC2626'}

for name, cost, wer_val, color, tier in models_plot:
    if wer_val is not None:
        ax.scatter(cost, wer_val, s=200, color=color, zorder=5, edgecolors='white', linewidth=1.5)
        ax.annotate(name, (cost, wer_val), textcoords="offset points",
                    xytext=(12, 4), fontsize=8.5, color=color, fontweight='bold')

ax.set_xlabel("Approximate Monthly Cost (USD) for ~1 hr audio", fontsize=11)
ax.set_ylabel("Word Error Rate — WER % (lower = better)", fontsize=11)
ax.set_title("Cost vs. Accuracy: Telugu ASR Models
(Bottom-Left = Best Value)", fontsize=13, fontweight='bold', color='#1a1a5e')
ax.invert_yaxis()   # lower WER is better, so put it at the top

patches = [mpatches.Patch(color=c, label=t) for t, c in tier_colors.items()]
ax.legend(handles=patches, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("cost_vs_accuracy.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved. Replace WER values with your actual results!")


### Step 3: Privacy Deep-Dive

When you call a paid API, your audio travels over the internet to a company's server.
Run the cell below to see what that means in practice.


In [ ]:
# Privacy flow visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#F8F9FF')

titles = ["Free/Local Model (e.g., Whisper)", "Paid API (e.g., Google Cloud)"]
colors = ['#16A34A', '#DC2626']

for ax, title, color in zip(axes, titles, colors):
    ax.set_xlim(0, 5); ax.set_ylim(0, 5); ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold', color=color)

# Left: local flow
ax = axes[0]
items = [(2.5, 4.3, "Your Audio"), (2.5, 3.1, "Your Device
(Colab/Laptop)"),
         (2.5, 1.9, "Transcript Output")]
for x, y, label in items:
    r = FancyBboxPatch((x-1.1, y-0.4), 2.2, 0.8,
                       boxstyle="round,pad=0.1", linewidth=2, edgecolor='#16A34A', facecolor='#DCFCE7')
    ax.add_patch(r)
    ax.text(x, y, label, ha='center', va='center', fontsize=9.5, color='#14532D', fontweight='bold')
for y1, y2 in [(3.9, 3.5), (2.7, 2.3)]:
    ax.annotate("", xy=(2.5, y2), xytext=(2.5, y1),
                arrowprops=dict(arrowstyle='->', color='#16A34A', lw=2))
ax.text(2.5, 1.1, "Data never leaves your machine
Full privacy", ha='center',
        fontsize=9, color='#16A34A', style='italic')

# Right: API flow
ax = axes[1]
items_right = [(2.5, 4.3, "Your Audio"), (2.5, 3.1, "Internet
(Encrypted)"),
               (2.5, 1.9, "Company Server
(Google/ElevenLabs)"), (2.5, 0.75, "Transcript Returned")]
ec_list = ['#DC2626', '#F97316', '#DC2626', '#6B7280']
fc_list = ['#FEE2E2', '#FFF7ED', '#FEE2E2', '#F9FAFB']
for (x, y, label), ec, fc in zip(items_right, ec_list, fc_list):
    r = FancyBboxPatch((x-1.2, y-0.4), 2.4, 0.8,
                       boxstyle="round,pad=0.1", linewidth=2, edgecolor=ec, facecolor=fc)
    ax.add_patch(r)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5, color=ec, fontweight='bold')
for y1, y2 in [(3.9, 3.5), (2.7, 2.3), (1.55, 1.15)]:
    ax.annotate("", xy=(2.5, y2), xytext=(2.5, y1),
                arrowprops=dict(arrowstyle='->', color='#DC2626', lw=2))
ax.text(2.5, 0.1, "Audio processed on their servers
Privacy policy applies", ha='center',
        fontsize=8.5, color='#DC2626', style='italic')

plt.tight_layout()
plt.savefig("privacy_flow.png", dpi=150, bbox_inches='tight')
plt.show()


### Step 4: Written Reflection (add your answers below)

**Question 1 — Cost vs. Quality:**
Did the paid or freemium tools produce noticeably better Telugu output than the free open-source models?
Was the difference worth it for a student project?

*Your answer:*

---

**Question 2 — Privacy Tradeoff:**
When you use a paid API, your audio is processed on someone else's server.
Why might this matter for a voice assistant used in someone's home?
Can you think of a case where a less-accurate free model is actually the *better* choice?

*Your answer:*

---

**Question 3 — Your Recommendation:**
Imagine you are advising a small nonprofit in Andhra Pradesh that wants to build a Telugu voice assistant
for farmers who cannot read. They have a budget of $0–$50/month and handle sensitive conversations.
Which model combination would you recommend, and why?

*Your answer:*
